# Перенос обучения на вырезках с витрины

Моя проверка учебного примера: создам папки тех же четырёх классов, получу признаки и сравню изменения параметров и состояния сети при двух шагах дообучения.

In [1]:
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torchvision import models,datasets,transforms
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from PIL import Image
torch.set_num_threads(2)
torch.manual_seed(0)
d=np.load('/Users/dmitriikormin/Documents/Projects/ds-course/site/data/showcase.npz')
root=Path('my-photos')
for i,name in enumerate(d['classes']):
    for k,j in enumerate(np.flatnonzero(d['y']==i)[:24]):
        folder=root/('train' if k<16 else 'validation')/name
        folder.mkdir(parents=True,exist_ok=True)
        Image.fromarray(d['X'][j]).save(folder/f'{j}.png')
gray=transforms.Compose([transforms.Grayscale(),transforms.Resize((32,32)),transforms.ToTensor()])
gray_train=datasets.ImageFolder(root/'train',transform=gray)
print(gray_train.class_to_idx,next(iter(DataLoader(gray_train,batch_size=8)))[0].shape)

{'круассан': 0, 'маффин': 1, 'печенье': 2, 'пусто': 3} torch.Size([8, 1, 32, 32])


In [2]:
w=models.ResNet18_Weights.DEFAULT
net=models.resnet18(weights=w)
net.fc=nn.Identity()
net.eval()
def loader(fold,bs=8):
    return DataLoader(datasets.ImageFolder(root/fold,transform=w.transforms()),batch_size=bs,shuffle=False)
def extract(fold):
    with torch.no_grad():
        pairs=[(net(xb).numpy(),yb.numpy()) for xb,yb in loader(fold)]
    return tuple(np.concatenate(cols) for cols in zip(*pairs))
etr,ytr=extract('train')
eva,yva=extract('validation')
lr=LogisticRegression(max_iter=2000).fit(etr,ytr)
print(etr.shape,eva.shape,lr.score(eva,yva))

(64, 512) (32, 512) 0.875


In [3]:
model=models.resnet18(weights=w)
model.fc=nn.Linear(model.fc.in_features,4)
model.requires_grad_(False)
model.fc.requires_grad_(True)
def count():
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print('голова',count(),'всего',sum(p.numel() for p in model.parameters()))
xb,yb=next(iter(loader('train')))
before=model.bn1.running_mean.clone()
conv_before=model.conv1.weight.clone()
opt=torch.optim.Adam(model.fc.parameters(),lr=.001)
model.train()
loss=nn.functional.cross_entropy(model(xb),yb)
opt.zero_grad()
loss.backward()
opt.step()
print('дельта весов свёртки',float((model.conv1.weight-conv_before).abs().max()))
print('дельта скользящего среднего BN',float((model.bn1.running_mean-before).abs().max()))
model.eval()
model.fc.train()
before=model.bn1.running_mean.clone()
opt.zero_grad()
nn.functional.cross_entropy(model(xb),yb).backward()
opt.step()
print('дельта при eval тела',float((model.bn1.running_mean-before).abs().max()))

голова 2052 всего 11178564
дельта весов свёртки 0.0
дельта скользящего среднего BN 0.289922297000885


дельта при eval тела 0.0


In [4]:
model.layer4.requires_grad_(True)
opt=torch.optim.Adam([{'params':model.fc.parameters(),'lr':1e-3},{'params':model.layer4.parameters(),'lr':1e-4}])
print('голова + верхний блок',count())
model.eval()
model.layer4.train()
model.fc.train()
for xb,yb in loader('train'):
    loss=nn.functional.cross_entropy(model(xb),yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
model.eval()
with torch.no_grad():
    matches=[model(xb).argmax(1)==yb for xb,yb in loader('validation')]
print('одна эпоха верхнего блока, валидация',torch.cat(matches).float().mean().item())

голова + верхний блок 8395780


одна эпоха верхнего блока, валидация 0.21875
